# Lab 2 – Orchestrate Rotisserie Chicken Agents

## Scenario
You will reuse the **RotisseriePlannerAgent** from Lab 1 and create three additional specialized agents. Then you will orchestrate all agents in a **sequential workflow** using the **Microsoft Agent Framework**.

### Workflow Pipeline
```
CoordinatorAgent → DemandForecasterAgent → CookingSchedulerAgent → WasteReductionAgent → RotisseriePlannerAgent
```

| Agent | Responsibility |
|-------|---------------|
| **CoordinatorAgent** | Structures user request into a brief for downstream agents |
| **DemandForecasterAgent** | Predicts daily/hourly demand using historical sales data |
| **CookingSchedulerAgent** | Creates batch cooking schedules aligned to demand peaks |
| **WasteReductionAgent** | Identifies waste risks and suggests real-time mitigations |
| **RotisseriePlannerAgent** | (Lab 1) Produces final consolidated cooking plan |

## Prerequisites
- Completed **Lab 1** (you have `AZURE_FOUNDRY_PROJECT_ENDPOINT` + `RotisseriePlannerAgent`)
- Python 3.10+
- Azure CLI authenticated: `az login`

## Step 1 — Install required packages

In [ ]:
%pip install azure-ai-projects --pre
%pip install azure-ai-agents --pre
%pip install azure-identity openai python-dotenv

# Microsoft Agent Framework
%pip install agent-framework --pre

## Step 2 — Load config and authenticate

In [ ]:
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

load_dotenv()

project_endpoint = os.environ["AZURE_FOUNDRY_PROJECT_ENDPOINT"]
model_id = os.getenv("AZURE_FOUNDRY_PROJECT_MODEL_ID", "gpt-4o-mini")

credential = DefaultAzureCredential()
project_client = AIProjectClient(endpoint=project_endpoint, credential=credential)
openai_client = project_client.get_openai_client()

print("Connected to project endpoint:", project_endpoint)
print("Model:", model_id)

## Step 3 — Verify the RotisseriePlannerAgent from Lab 1

In [ ]:
planner_agent_name = os.getenv("ROTISSERIE_PLANNER_AGENT_NAME", "RotisseriePlannerAgent")
planner_agent = project_client.agents.get(agent_name=planner_agent_name)
print("Found RotisseriePlannerAgent:", getattr(planner_agent, 'name', None))

## Step 4 — Upload data files to vector store

Create a shared vector store for the new agents to use.

In [ ]:
# Create vector store for the orchestration agents
vector_store = openai_client.vector_stores.create(name="zava_rotisserie_orchestration")
print(f"Vector store created (id: {vector_store.id})")

data_files = [
    "../data/chicken_hourly_store_sales.json",
    "../data/chicken_daily_orders_financials.json"
]

for file_path in data_files:
    file = openai_client.vector_stores.files.upload_and_poll(
        vector_store_id=vector_store.id, file=open(file_path, "rb")
    )
    print(f"Uploaded {file_path} (id: {file.id})")

tools = [{"type": "file_search", "vector_store_ids": [vector_store.id]}]

## Step 5 — Create the CoordinatorAgent

This agent structures user requests into actionable briefs for downstream agents.

In [ ]:
from azure.ai.projects.models import PromptAgentDefinition

coordinator_instructions = """You are the CoordinatorAgent for Zava rotisserie chicken operations.
Convert the user request into a structured brief for downstream agents.
Output a JSON object with these fields:
- goal: what the user wants to achieve
- target_day: the day being planned for
- constraints: any constraints mentioned (weather, events, budget)
- data_notes: which data sources to consult
- required_outputs: what the final answer should include"""

coordinator = project_client.agents.create_version(
    agent_name="ChickenCoordinatorAgent",
    definition=PromptAgentDefinition(
        model=model_id,
        instructions=coordinator_instructions,
    ),
    description="Structures user requests into briefs for the rotisserie planning pipeline.",
)

print("Created CoordinatorAgent:", getattr(coordinator, 'name', None))

## Step 6 — Create the DemandForecasterAgent

Predicts how many chickens will sell, broken down by hour.

In [ ]:
forecaster_instructions = """You are the DemandForecasterAgent for Zava rotisserie operations.
Given historical sales data and context from the coordinator:
1. Predict total chickens that will sell for the target day
2. Break the forecast into hourly buckets (06:00-21:00)
3. Flag any unusual patterns (weather impact, day-of-week trends)
4. Provide confidence level (high/medium/low) for your forecast
5. Reference specific historical data points to justify predictions

Use the sales data files to find patterns by day of week.
Mondays/Tuesdays are typically low (150-170 chickens).
Fridays/Saturdays are peak (220-240 chickens).
Output your forecast as structured JSON."""

forecaster = project_client.agents.create_version(
    agent_name="DemandForecasterAgent",
    definition=PromptAgentDefinition(
        model=model_id,
        instructions=forecaster_instructions,
        tools=tools,
    ),
    description="Forecasts hourly and daily chicken demand from historical sales.",
)

print("Created DemandForecasterAgent:", getattr(forecaster, 'name', None))

## Step 7 — Test the DemandForecasterAgent

In [ ]:
from IPython.display import display, Markdown

forecaster_name = getattr(forecaster, "name", "DemandForecasterAgent")
question = "Forecast demand for next Friday. How many chickens per hour?"

response = openai_client.responses.create(
    input=question,
    extra_body={"agent": {"type": "agent_reference", "name": forecaster_name}},
)

display(Markdown(response.output_text))

## Step 8 — Create the CookingSchedulerAgent

Takes demand forecast and creates batch cooking schedules.

In [ ]:
scheduler_instructions = """You are the CookingSchedulerAgent for Zava rotisserie operations.
Given the demand forecast from DemandForecasterAgent:
1. Create a batch cooking schedule with specific times and quantities
2. Each batch takes 90 minutes to cook — plan accordingly
3. Chickens stay fresh for ~2 hours after cooking
4. Plan batches to be ready 15 minutes BEFORE peak demand
5. Include a small buffer (10-15%) for unexpected demand
6. Minimize batches during low-demand periods (before 10am, after 8pm)

Output a schedule like:
- Batch 1: Start cooking at 09:30, quantity: 25, ready by 11:00 (for lunch rush)
- Batch 2: Start cooking at 11:00, quantity: 20, ready by 12:30 (lunch continuation)
etc.

Output as structured JSON with batch details."""

scheduler = project_client.agents.create_version(
    agent_name="CookingSchedulerAgent",
    definition=PromptAgentDefinition(
        model=model_id,
        instructions=scheduler_instructions,
        tools=tools,
    ),
    description="Creates batch cooking schedules aligned to demand peaks.",
)

print("Created CookingSchedulerAgent:", getattr(scheduler, 'name', None))

## Step 9 — Create the WasteReductionAgent

Analyzes waste patterns and suggests real-time adjustments.

In [ ]:
waste_instructions = """You are the WasteReductionAgent for Zava rotisserie operations.
Given the cooking schedule and historical waste data:
1. Identify time slots with highest waste risk
2. Suggest specific adjustments to reduce waste
3. Propose real-time decision rules for store associates:
   - "If fewer than X sold by Y time, skip next batch"
   - "If more than X in warmer at Z time, discount by $2"
4. Calculate potential savings from your recommendations
5. Flag end-of-day waste risk and mitigation strategies

Historical context: Zava averages ~4.7% waste rate.
Goal is to reduce waste below 3% while avoiding stockouts.

Output as structured JSON with waste_risks and mitigations."""

waste_agent = project_client.agents.create_version(
    agent_name="WasteReductionAgent",
    definition=PromptAgentDefinition(
        model=model_id,
        instructions=waste_instructions,
        tools=tools,
    ),
    description="Identifies waste risks and suggests real-time adjustments.",
)

print("Created WasteReductionAgent:", getattr(waste_agent, 'name', None))

## Step 10 — Build the Sequential Workflow

Orchestrate all five agents in a pipeline:

```
Coordinator → DemandForecaster → CookingScheduler → WasteReduction → RotisseriePlanner
```

In [ ]:
from typing import cast
from agent_framework import AgentResponse
from agent_framework.azure import AzureAIAgentClient
from agent_framework_orchestrations import SequentialBuilder

# Create agent clients
coordinator_client = AzureAIAgentClient(
    model_deployment_name=model_id, credential=credential,
    project_endpoint=project_endpoint, agent_name="ChickenCoordinatorAgent"
)
forecaster_client = AzureAIAgentClient(
    model_deployment_name=model_id, credential=credential,
    project_endpoint=project_endpoint, agent_name="DemandForecasterAgent"
)
scheduler_client = AzureAIAgentClient(
    model_deployment_name=model_id, credential=credential,
    project_endpoint=project_endpoint, agent_name="CookingSchedulerAgent"
)
waste_client = AzureAIAgentClient(
    model_deployment_name=model_id, credential=credential,
    project_endpoint=project_endpoint, agent_name="WasteReductionAgent"
)
planner_client = AzureAIAgentClient(
    model_deployment_name=model_id, credential=credential,
    project_endpoint=project_endpoint, agent_name="RotisseriePlannerAgent"
)

# Convert to Agent Framework agents
coord_agent = coordinator_client.as_agent(name="ChickenCoordinatorAgent")
forecast_agent = forecaster_client.as_agent(name="DemandForecasterAgent")
schedule_agent = scheduler_client.as_agent(name="CookingSchedulerAgent")
waste_ag = waste_client.as_agent(name="WasteReductionAgent")
planner_ag = planner_client.as_agent(name="RotisseriePlannerAgent")

print("All agent clients created successfully")

## Step 11 — Run the workflow

In [ ]:
prompt = (
    "What should I cook tomorrow (Friday) and when? "
    "Weather forecast says partly cloudy, 65°F. "
    "Give me a full cooking plan with batch times, quantities, "
    "waste reduction tips, and real-time adjustment rules."
)

# Build sequential workflow
workflow = SequentialBuilder(
    participants=[
        coord_agent,
        forecast_agent,
        schedule_agent,
        waste_ag,
        planner_ag
    ]
).build()

# Run as a single agent
agent = workflow.as_agent(name="rotisserie_workflow")
agent_response = await agent.run(prompt)

print("=== WORKFLOW OUTPUTS ===")
if agent_response.messages:
    print("\n===== Conversation =====")
    for i, msg in enumerate(agent_response.messages, start=1):
        name = msg.author_name or msg.role
        print(f"{'-' * 60}")
        print(f"{i:02d} [{name}]")
        print(msg.text)

## Step 12 — Try another scenario

Test with a different day and conditions.

In [ ]:
prompt2 = (
    "Plan for Monday. It's expected to rain all day, temperature around 50°F. "
    "Mondays are usually our slowest day. "
    "How can we minimize waste while still having enough chickens ready?"
)

agent_response2 = await agent.run(prompt2)

print("=== RAINY MONDAY PLAN ===")
if agent_response2.messages:
    for i, msg in enumerate(agent_response2.messages, start=1):
        name = msg.author_name or msg.role
        print(f"{'-' * 60}")
        print(f"{i:02d} [{name}]")
        print(msg.text)

## (Optional) Handoff Scenarios — Home Exercise

Consider implementing these conditional handoff patterns:

1. **Stockout Alert Handoff**: If DemandForecaster predicts demand > 95% capacity, skip WasteReduction and go directly to emergency cooking plan

2. **High Waste Alert Handoff**: If WasteReductionAgent detects >8% projected waste, loop back to CookingScheduler for re-optimization

3. **Weather Emergency Handoff**: If weather is extreme (snow/heat wave), Coordinator bypasses normal flow and triggers a simplified low-risk plan

4. **Budget Threshold Handoff**: If projected costs exceed daily budget, WasteReductionAgent escalates to Coordinator for human approval

## Cleanup (optional)

In [ ]:
# Uncomment to delete agents created in this lab
# project_client.agents.delete(agent_name="ChickenCoordinatorAgent")
# project_client.agents.delete(agent_name="DemandForecasterAgent")
# project_client.agents.delete(agent_name="CookingSchedulerAgent")
# project_client.agents.delete(agent_name="WasteReductionAgent")

## Validation Checklist
- ✅ Reused RotisseriePlannerAgent from Lab 1
- ✅ Created CoordinatorAgent, DemandForecasterAgent, CookingSchedulerAgent, WasteReductionAgent
- ✅ Built sequential workflow orchestrating all 5 agents
- ✅ Workflow produces a complete cooking plan with batch schedules
- ✅ Tested with multiple scenarios (peak day, slow day, weather variations)